# ML Pipeline — Exercises

**Companion to deck 05 (5-step mantra).** Build a Titanic classifier end-to-end. Same shape as every camp baseline.

<a href="https://colab.research.google.com/github/Petkub/MachineLearningLab/blob/main/HTMLSlides/decks/05-ml-pipeline/exercises.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

df = sns.load_dataset('titanic').rename(columns={
    'pclass': 'Pclass', 'sex': 'Sex', 'age': 'Age',
    'fare': 'Fare', 'survived': 'Survived',
})
# encode + impute
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
df['Age'] = df['Age'].fillna(df['Age'].mean())
df = df.dropna(subset=['Fare'])
print(df.shape)
df.head()

---
## Problem 01 — separate X and y

Tasks:
1. Build `X` — features `['Pclass', 'Sex', 'Age', 'Fare']` from `df`.
2. Build `y` — the `Survived` column.
3. Print shapes.

In [ ]:
# TODO
X = ...
y = ...

print('X:', X.shape, '| y:', y.shape)

In [ ]:
assert X.shape[1] == 4, f'X should have 4 columns, has {X.shape[1]}'
assert 'Survived' not in X.columns, 'target leaked into X — remove it'
assert len(X) == len(y), 'X and y have different row counts'
print('Q1 ok')

<details><summary>Hint</summary>

`X = df[['Pclass', 'Sex', 'Age', 'Fare']]` and `y = df['Survived']`. Don't put 'Survived' into X — that's a data leak.
</details>

---
## Problem 02 — split

Hold 20% of rows for test. Use `random_state=42`. Keep class ratios balanced with `stratify=y`.

In [ ]:
# TODO
X_train, X_test, y_train, y_test = ...

print('train:', X_train.shape, '| test:', X_test.shape)
print('train survival rate:', y_train.mean().round(3))
print('test  survival rate:', y_test.mean().round(3))

In [ ]:
assert abs(len(X_test) / (len(X_train) + len(X_test)) - 0.2) < 0.01, 'test_size should be ~0.2'
assert abs(y_train.mean() - y_test.mean()) < 0.02, 'class ratios drift — did you use stratify=y?'
print('Q2 ok')

<details><summary>Hint</summary>

`train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)`.
</details>

---
## Problem 03 — fit + predict + evaluate

Train a `LogisticRegression` (use `max_iter=1000` so it converges). Predict on the test set. Print accuracy.

In [ ]:
# TODO
model = ...
# fit on train
# predict on test
preds = ...
acc = ...

print('accuracy:', round(acc, 3))

In [ ]:
assert 0.70 < acc < 0.90, f'accuracy out of expected range: {acc}'
assert len(preds) == len(y_test), 'preds length mismatch'
print('Q3 ok — LogReg accuracy:', round(acc, 3))

<details><summary>Hint</summary>

`model = LogisticRegression(max_iter=1000)`, then `model.fit(X_train, y_train)`, `preds = model.predict(X_test)`, `acc = accuracy_score(y_test, preds)`.
</details>

---
## Problem 04 — swap models without changing the pipeline

Train and score three models: LogReg, RandomForest, KNN. Save accuracies in a dict `scores`.

**Goal:** prove to yourself that only one line changes per model.

In [ ]:
# TODO
scores = {}

# for each model: fit on train, predict on test, record accuracy in scores
# use these:
#   LogisticRegression(max_iter=1000)
#   RandomForestClassifier(n_estimators=100, random_state=42)
#   KNeighborsClassifier(n_neighbors=5)

for name, score in scores.items():
    print(f'{name:25s} {score:.3f}')

In [ ]:
assert set(scores.keys()) >= {'logreg', 'rf', 'knn'}, 'scores dict must have keys: logreg, rf, knn'
for name, s in scores.items():
    assert 0.6 < s < 0.95, f'{name} score out of range: {s}'
print('Q4 ok — winner:', max(scores, key=scores.get))

<details><summary>Hint</summary>

Define a list of `(name, model)` tuples, loop, fit, predict, store. Same X_train, y_train, X_test, y_test for all.
</details>

---
## Problem 05 — beyond accuracy

Use the best model from Q4. Compute confusion matrix, precision, recall, F1.

Question to answer in your head: which mistake type is more common — predicting *survived* when they didn't (FP), or *died* when they did (FN)?

In [ ]:
# TODO
best = ...   # winning model from Q4 — refit if needed, or reuse preds
best_preds = ...

cm = confusion_matrix(y_test, best_preds)
print('confusion matrix:\n', cm)
print('precision:', round(precision_score(y_test, best_preds), 3))
print('recall:   ', round(recall_score(y_test, best_preds), 3))
print('f1:       ', round(f1_score(y_test, best_preds), 3))

In [ ]:
assert cm.shape == (2, 2), 'confusion matrix should be 2x2 for binary'
tn, fp, fn, tp = cm.ravel()
print(f'TN={tn} FP={fp} FN={fn} TP={tp}')
print('Q5 ok')

---
## Problem 06 — accuracy trap

Build a baseline model that **always predicts the majority class** (everyone died). Compute its accuracy.

If your fancy model only beats this by 5 points, your fancy model isn't fancy.

In [ ]:
# TODO — no sklearn needed, just numpy
majority = ...   # most common class in y_train (0 or 1)
dummy_preds = ...   # array of length len(y_test), all = majority
dummy_acc = ...

print('majority class:', majority)
print('dummy accuracy:', round(dummy_acc, 3))
print('your best model:', round(max(scores.values()), 3))
print('lift over dummy:', round(max(scores.values()) - dummy_acc, 3))

In [ ]:
assert majority in (0, 1)
assert len(dummy_preds) == len(y_test)
assert max(scores.values()) > dummy_acc, 'your model should beat the dummy'
print('Q6 ok — your lift:', round(max(scores.values()) - dummy_acc, 3))

<details><summary>Hint</summary>

`majority = y_train.mode()[0]`. `dummy_preds = np.full(len(y_test), majority)`. `dummy_acc = accuracy_score(y_test, dummy_preds)`.
</details>

---
## Solutions

<details><summary>Show all solutions</summary>

```python
# Q1
X = df[['Pclass', 'Sex', 'Age', 'Fare']]
y = df['Survived']

# Q2
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Q3
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
preds = model.predict(X_test)
acc = accuracy_score(y_test, preds)

# Q4
candidates = {
    'logreg': LogisticRegression(max_iter=1000),
    'rf':     RandomForestClassifier(n_estimators=100, random_state=42),
    'knn':    KNeighborsClassifier(n_neighbors=5),
}
scores = {}
for name, m in candidates.items():
    m.fit(X_train, y_train)
    scores[name] = accuracy_score(y_test, m.predict(X_test))

# Q5
best_name = max(scores, key=scores.get)
best = candidates[best_name]
best_preds = best.predict(X_test)

# Q6
majority = y_train.mode()[0]
dummy_preds = np.full(len(y_test), majority)
dummy_acc = accuracy_score(y_test, dummy_preds)
```
</details>